In [11]:
import polars as pl
import os
from PIL import Image
from transformers import Siglip2Tokenizer, Siglip2ImageProcessorFast

base_csv_path = "../../datasets/google-landmark"
base_img_train_path = "../../datasets/google-landmark/train-img"
base_img_index_path = "../../datasets/google-landmark/index-img"

In [12]:
df_predicted = pl.read_csv("../../datasets/google-landmark/raw-csv/index_predicted.csv")
scores = df_predicted["confidence"].to_numpy()

In [13]:
df_predicted.filter(pl.col("confidence") >= 0.01).sort("confidence")

id,landmark_id,category,confidence
str,i64,str,f64
"""fd0a4ab42fe2f8d3""",41368,"""church_cathedral""",0.01001
"""5d6c47c903433e91""",39005,"""church_cathedral""",0.01001
"""c556cbadb012954b""",100613,"""castle_fortress""",0.01001
"""f12bd40f9e755f5f""",18741,"""church_cathedral""",0.01001
"""64d404550a8235c5""",41610,"""ruins""",0.01001
…,…,…,…
"""e14138cf14d67589""",95395,"""arch_gate""",0.997559
"""e5af9af43973cc43""",17322,"""arch_gate""",0.998047
"""6e96f7486260ad0c""",35288,"""railway_station""",0.998535


In [20]:
import numpy as np

percentiles = [1, 5, 10, 25, 50, 75, 90, 95, 99]
for p in percentiles:
    print(f"p{p:3d}: {np.percentile(scores, p):.4f}")

print(f"\nmean:   {scores.mean():.4f}")
print(f"median: {np.median(scores):.4f}")

p  1: 0.0000
p  5: 0.0000
p 10: 0.0000
p 25: 0.0000
p 50: 0.0005
p 75: 0.0124
p 90: 0.1038
p 95: 0.2815
p 99: 0.7505

mean:   0.0434
median: 0.0005


In [10]:
# See how many pass at various thresholds
for t in [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40]:
    n = (scores > t).sum()
    print(f"> {t:.2f}: {n:>7,}  ({100*n/len(scores):.1f}%)")

> 0.10:  77,991  (10.2%)
> 0.15:  61,160  (8.0%)
> 0.20:  50,161  (6.6%)
> 0.25:  42,352  (5.6%)
> 0.30:  35,974  (4.7%)
> 0.35:  30,926  (4.1%)
> 0.40:  26,587  (3.5%)


In [6]:
df_train = pl.read_csv(os.path.join(base_csv_path, "train_ref_predicted_siglip_filtered.csv"))
df_index = pl.read_csv(os.path.join(base_csv_path, "index_ref_predicted_siglip_filtered.csv"))

In [7]:
df_train = df_train.filter(pl.col("confidence") >= 0.35)
df_index = df_index.filter(pl.col("confidence") >= 0.35)

In [ ]:
df_train.write_csv("../../datasets/geocir-triplet/prep_gld_train.csv")
df_index.write_csv("../../datasets/geocir-triplet/prep_gld_index.csv")

In [ ]:
import sys
sys.path.append("..")
from src.utils import read_index

train_db, train_meta = read_index("../../datasets/geocir-triplet/prep_gld_train")
index_db, index_meta = read_index("../../datasets/geocir-triplet/prep_gld_index")

train_meta = train_meta["metadata"]
index_meta = index_meta["metadata"]

In [17]:
df_train.group_by(["category", "country"]).len().filter(pl.col("len") >= 100)

category,country,len
str,str,u32
"""museum_theater""","""France""",254
"""arch_gate""","""Austria""",165
"""temple_shrine""","""China""",2557
"""monument_statue""","""China""",256
"""monument_statue""","""Germany""",585
…,…,…
"""mosque""","""Romania""",111
"""railway_station""","""Spain""",148
"""arch_gate""","""Iran""",197
